The code in this notebook is based on the original PyTorch3D tutorial: https://pytorch3d.org/tutorials/render_textured_meshes

---


# Render a textured mesh & verify coordinate systems (PyTorch3D ↔ OpenCV)

This notebook will:

1. **Load** a textured mesh from an `.obj`.
2. **Configure** a PyTorch3D renderer (camera, rasterizer, shader, lights).
3. **Render** the mesh and **tune** settings (lighting, materials, camera pose).
4. **Batch-render** multiple viewpoints efficiently using PyTorch3D’s batched API.
5. **Validate coordinates** with small tests/overlays to understand PyTorch3D’s camera frame and how it relates to OpenCV.




# ⚙️ Notebook Environment Selection (Manual Switch)

There is no reliable way to automatically detect whether this notebook is running:

- in **Colab (web UI)**  
- or in **VS Code connected to a remote Colab runtime**

Because of this limitation, we use a **manual flag** to explicitly define the environment.

---

## 🛠️ How to use

In the next code cell, set:

```python
RUNNING_IN_VSCODE = True

In [9]:
import os

#@title Check the box to indicate the IDE environment you want to use:
#@markdown
#@markdown If editing the code outside Colab, just change
#@markdown `RUNNING_IN_VSCODE = False` to `RUNNING_IN_VSCODE = True`.


# ── Set this manually ──────────────────────
RUNNING_IN_VSCODE = True          #@param {type:"boolean"}
# ───────────────────────────────────────────

os.environ['NOTEBOOK_ENV'] = 'vscode' if RUNNING_IN_VSCODE else 'colab'
print("Environment:", os.environ['NOTEBOOK_ENV'])





from IPython.display import HTML, display
import os

env = os.environ.get("NOTEBOOK_ENV", "unknown")

if env == "vscode":
    display(HTML("""
    <div style="
        padding: 14px 18px;
        margin-top: 8px;
        border-left: 6px solid #f59e0b;
        border-radius: 8px;
        background-color: #fff7ed;
        box-shadow: 0 3px 10px rgba(0,0,0,0.10);
        font-family: 'Helvetica Neue', Arial, sans-serif;
        font-size: 15px;
        color: #7c2d12;
    ">
        ⚠️ <b>VS Code Mode Detected</b><br>
        Colab Secrets are <b>not available</b> in this environment.<br>
        You may need to manually provide credentials (e.g., GitHub token).
    </div>
    """))
else:
    display(HTML("""
    <div style="
        padding: 14px 18px;
        margin-top: 8px;
        border-left: 6px solid #f59e0b;
        border-radius: 8px;
        background-color: #fff7ed;
        box-shadow: 0 3px 10px rgba(0,0,0,0.10);
        font-family: 'Helvetica Neue', Arial, sans-serif;
        font-size: 15px;
        color: #7c2d12;
    ">
        ⚠️ <b>Running in Web-Based Colab</b><br>
        GitHub credentials (i.e., personal token) is stored in Colab Secrets in this environment.<br>
    </div>
    """))

Environment: vscode


# 🔐 Secure GitHub Token Prompt (VS Code + Remote Colab)

When running this notebook from **VS Code connected to a remote Colab runtime**,  
Colab Secrets are not accessible. In this case, we must manually provide the GitHub token.

Instead of hardcoding the token (unsafe), we use a secure prompt.

---

## 📌 Instructions

1.  Open your Notion workspace (**This step is in my case specifically**):

    - Token location (my case): `Notion → Main page → Programming → GitHub Token`

2. Copy your GitHub Personal Access Token.

3. Run the code cell below.

4. Paste the token when prompted (input will be hidden).

---

## 🔒 Security Notes

- The token is **not stored in the notebook**
- The token is **not printed in outputs**
- The token exists **only in memory during runtime**
- It is safe to commit this notebook to GitHub

⚠️ Never hardcode your token in a cell or commit it to the repository.

---

## 💡 When is this needed?

- ✅ VS Code + remote Colab
- ✅ Local Jupyter
- ❌ Not needed in standard Colab (web UI with Secrets enabled)

---

## ✅ What this does

The next cell will:
- Prompt you securely for your token
- Store it in `os.environ["GH_TOKEN"]`
- Make it available to any code that needs GitHub authentication

In [10]:
#@title Secure GitHub Token Setup (VS Code + Colab Secrets)

import os
from getpass import getpass
from IPython.display import HTML, display

# ------------------------------------------------------------
# CASE 1: Running in VS Code → prompt user
# ------------------------------------------------------------
if RUNNING_IN_VSCODE:

    if not os.environ.get("GH_TOKEN"):
        token = getpass("Enter your GitHub Personal Access Token: ")
        os.environ["GH_TOKEN"] = token

    display(HTML("""
    <div style="
        padding: 12px 16px;
        margin: 8px 0;
        border-radius: 10px;
        background: linear-gradient(135deg, #f5f7fa, #e4ecf7);
        border: 1px solid #c9d7ea;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        font-family: Arial, sans-serif;
        font-size: 15px;
        color: #1f3b5b;
    ">
        <strong>GH_TOKEN set via manual input.</strong>
    </div>
    """))

# ------------------------------------------------------------
# CASE 2: Running in Colab → use Secrets
# ------------------------------------------------------------
else:
    try:
        from google.colab import userdata

        token = userdata.get("GH_TOKEN")  # must match secret name
        os.environ["GH_TOKEN"] = token

        display(HTML("""
        <div style="
            padding: 12px 16px;
            margin: 8px 0;
            border-radius: 10px;
            background: #ecfdf5;
            border: 1px solid #a7f3d0;
            font-family: Arial, sans-serif;
            font-size: 14px;
            color: #065f46;
        ">
            <strong>GH_TOKEN loaded from Colab Secrets.</strong>
        </div>
        """))

    except Exception as e:
        display(HTML(f"""
        <div style="
            padding: 12px 16px;
            margin: 8px 0;
            border-radius: 10px;
            background: #fef2f2;
            border: 1px solid #fecaca;
            font-family: Arial, sans-serif;
            font-size: 14px;
            color: #7f1d1d;
        ">
            ⚠️ <strong>Failed to load GH_TOKEN from Colab Secrets.</strong><br>
            Make sure a secret named <code>GH_TOKEN</code> exists.
        </div>
        """))

In [11]:
#@title Select your Colab GPU (or Auto)
colab_instance = "Auto"  # @param ["Auto", "A100", "T4", "L4", "CPU"]

import subprocess, re

def detect_gpu():
    try:
        out = subprocess.check_output("nvidia-smi -L", shell=True).decode()
        if "A100" in out:
            return "A100"
        if "T4" in out:
            return "T4"
        if "L4" in out:
            return "L4"
    except:
        pass
    return "CPU"

if colab_instance == "Auto":
    colab_instance = detect_gpu()




from IPython.display import HTML, display

display(HTML(f"""
<div style="
    padding: 12px 16px;
    margin: 8px 0;
    border-radius: 10px;
    background: linear-gradient(135deg, #f5f7fa, #e4ecf7);
    border: 1px solid #c9d7ea;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08);
    font-family: Arial, sans-serif;
    font-size: 15px;
    color: #1f3b5b;
">
    <span style="font-size: 18px; margin-right: 8px;">🔧</span>
    <strong>Using GPU type:</strong> {colab_instance}
</div>
"""))

# 🔄 Update Repository (Pull Latest Changes)

Run the cell below whenever updates have been made to the repository that contains  
the **library / source files used by this notebook**.

---

## 📌 When should this be used?

- After modifying code in the repository (e.g., `.py` library files)
- After pulling new changes from another machine
- When switching branches or syncing with remote updates

---

## 🛠️ What this does

- Executes a `git pull` inside the repository: `/content/pytorch3d_rendering`
- Fetches and merges the latest changes from the remote repository
- Displays the output directly in the notebook

---

## ▶️ How to use

1. Click the **"Git Pull"** button below  
2. Wait for the output to confirm the update  
3. Re-run any affected cells if needed  

---

## ⚠️ Notes

- This does **not reset** or overwrite uncommitted local changes  
- If conflicts occur, they must be resolved manually  
- Make sure the repository is already cloned before using this  

---

## 💡 Tip

If you updated core library code, it is recommended to:

- Restart the runtime **or**
- Reload modules (if using dynamic imports)

to ensure the latest code is being used.
  

In [12]:
#@title Update Repository (Pull Latest Changes)

import os
import ipywidgets as widgets
from IPython.display import display, HTML
import subprocess

REPO_PATH = "/content/pytorch3d_rendering"

button = widgets.Button(
    description="Git Pull",
    button_style="primary",
    icon="download"
)
output = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()

        if not os.path.isdir(os.path.join(REPO_PATH, ".git")):
            display(HTML(f"""
            <div style="
                padding: 14px 18px;
                margin-top: 8px;
                border-left: 6px solid #f59e0b;
                border-radius: 8px;
                background-color: #fff7ed;
                box-shadow: 0 3px 10px rgba(0,0,0,0.10);
                font-family: 'Helvetica Neue', Arial, sans-serif;
                font-size: 15px;
                color: #7c2d12;
            ">
                ⚠️ <b>Repository Not Found</b><br>
                The repository <code>{REPO_PATH}</code> has <b>not been cloned yet</b>.<br><br>
                It will be cloned in a downstream cell.<br>
                After that, if the repository files change, return to this cell and click <b>Git Pull</b> to fetch the latest updates and refresh the local files.
            </div>
            """))
            return

        result = subprocess.run(
            ["git", "-C", REPO_PATH, "pull"],
            capture_output=True,
            text=True
        )

        print(result.stdout or result.stderr)

button.on_click(on_click)
display(button, output)

Button(button_style='primary', description='Git Pull', icon='download', style=ButtonStyle())

Output()

---
# ⚙️ Setting up
⚠️ **This section has some user-input values that might need to be set for different configurations and datasets**. Expland the subsection on UI to see the input widgets. ⚠️

## 🧰 Initial Imports, Clone Repositories, and Mount Google Drive


In [13]:
import os
import cv2
import torch

# OpenCV
cv2.setNumThreads(8)

# PyTorch
torch.set_num_threads(8)


In [14]:
from IPython.display import HTML, display

def show_message(message, title=None, level="info"):
    """
    Display a styled HTML message in a notebook.

    Args:
        message (str): Main message body (can include HTML).
        title (str, optional): Bold title.
        level (str): One of ["info", "success", "warning", "error"].
    """

    styles = {
        "info": {
            "bg": "linear-gradient(135deg, #f5f7fa, #e4ecf7)",
            "border": "#c9d7ea",
            "color": "#1f3b5b",
            "icon": "ℹ️"
        },
        "success": {
            "bg": "#ecfdf5",
            "border": "#a7f3d0",
            "color": "#065f46",
            "icon": "✅"
        },
        "warning": {
            "bg": "#fff7ed",
            "border": "#f59e0b",
            "color": "#7c2d12",
            "icon": "⚠️"
        },
        "error": {
            "bg": "#fef2f2",
            "border": "#ef4444",
            "color": "#7f1d1d",
            "icon": "❌"
        }
    }

    s = styles.get(level, styles["info"])

    title_html = f"<strong>{title}</strong><br>" if title else ""

    display(HTML(f"""
    <div style="
        padding: 12px 16px;
        margin: 8px 0;
        border-radius: 10px;
        background: {s['bg']};
        border-left: 6px solid {s['border']};
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        font-family: Arial, sans-serif;
        font-size: 15px;
        color: {s['color']};
    ">
        {s['icon']} {title_html}
        {message}
    </div>
    """))

In [15]:

#  ---------------------------- IMPORTS -----------------------------------------
# Stdlib
import os
import sys
import math
import shutil
from pathlib import Path
from typing import Optional, Tuple, Literal, Dict, Any

# Third-party
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import imageio
import requests
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm.notebook import tqdm
from skimage import img_as_ubyte


# Get device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# import traceback
# def show_error(e):
#     print(f"\033[91m❌ Exception: {e}\033[0m")
#     print(traceback.format_exc())



ModuleNotFoundError: No module named 'imageio'

In [ ]:
!pip --quiet install ipython-autotime
%load_ext autotime

### 📂 Clone Repository & 🔑 Mount Google Drive  & Install PyTorch3D/dependencies

Clone the repository and mount **Google Drive** (requires user interaction).  
This will also set up the environment and install the necessary libraries.


In [ ]:
# # **Set name and email for github cloning**
# !git config --global user.name git_username
# !git config --global user.email git_email

In [ ]:
def gh_clone(user, repo, branch="main", token_key="GH_TOKEN"):
    import os
    import subprocess
    from pathlib import Path

    token = os.environ.get(token_key, "")
    if not token:
        raise ValueError(f"Environment variable {token_key} is not set.")

    url = f"https://{token}@github.com/{user}/{repo}.git"
    repo_dir = Path("/content") / repo

    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", branch, "--single-branch", url, str(repo_dir)],
            check=True
        )
    else:
        print(f"Repo already exists: {repo_dir}")

    subprocess.run(
        ["git", "-C", str(repo_dir), "remote", "set-url", "origin", url],
        check=True
    )

    return str(repo_dir)

#### 🔽 Mount google drive

In [ ]:
# Set this to True if you want to mount gdrive
mount_gdrive = True

In [ ]:
import os

from google.colab import drive
from google.colab import auth

# auth.authenticate_user()

local_path = os.getcwd()
print("Current local path:", local_path)

# Mount google drive if using Colab
if 'google.colab' in str(get_ipython()):
    # print('Running on CoLab')
    show_message("Running on CoLab.")
    local_path = "/content/"
    from google.colab import drive
    if mount_gdrive:
        if mount_gdrive:
            drive.mount('/content/drive', force_remount=True)
else:
    # print('Not running on CoLab')
    show_message("Not running on CoLab.")

os.chdir(local_path)



##  🎛️ User-input (Expand this section to see/edit fields) ⚠️


⚠️ <b>Attention:</b> Replace the information with your GitHub email and username.


⚠️ <b>Attention:</b> Press enter or run cells to accept default values.
</div>


In [ ]:
#@title Settings for GitHub Access

import os
import sys
import subprocess

# Set name and email for github cloning
git_username = "eraldoribeiro"   #@param {type:"string"}
git_email = "eribeiro@fit.edu"   #@param {type:"string"}

repository_name = "pytorch3d_rendering"      #@param {type:"string"}
organization_name = "ribeiro-computer-vision"   #@param {type:"string"}
branch_name = "main"          #@param {type:"string"}

# -------------------------------------------------
# Set token once per session before running this:
# os.environ["GH_TOKEN"] = "your_token_here"
# -------------------------------------------------
if not os.environ.get("GH_TOKEN"):
    raise ValueError(
        'GH_TOKEN is not set.\n'
        'Run first:\n'
        'os.environ["GH_TOKEN"] = "your_token_here"'
    )

# Set git identity
subprocess.run(["git", "config", "--global", "user.name", git_username], check=True)
subprocess.run(["git", "config", "--global", "user.email", git_email], check=True)

# Clone or update repo into /content
repo_name = gh_clone(
    user=organization_name,
    repo=repository_name,
    branch=branch_name
)

# Verify clone
if os.path.exists(repo_name):
    # print(f"✅ Repository '{repo_name}' successfully cloned!")
    show_message(f"Repository '{repo_name}' successfully cloned!", level="success")
else:
    print(f"❌ Repository '{repo_name}' not found.")

# Add /content to Python path
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

print("\nTop sys.path entries:")
for p in sys.path[:5]:
    print(" ", p)

In [ ]:
#@title Path to Dropbox shared directory containing the PyTorch3D (pre-built) wheels

# Set name and email for github cloning using #@param
dropbox_link = "https://www.dropbox.com/scl/fo/5euku4a49i1emwq172nx1/AB7S9-l5x9BkGIc6LvHNAeY?rlkey=9k2o2snx1faaq94miepxif66b&dl=0" #@param {type:"string"}


## ⚙️ Install Pytorch3D

### PyTorch3D in Colab

Modules `torch` and `torchvision` are required. If `pytorch3d` is not installed, install it using the following cell. Here, I modified to install PyTorch3D from my own pre-built wheel. Using our own pytorch3d wheel allows for faster installation as installing PyTorch3D from source takes several minutes to complete.

**WARNING: If the PyTorch3D installation from the current wheel fails, create another one!!!**

PyTorch3D takes a long time to install from source in Colab. Instead of installing from source everytime an Colab instance is started, this notebook uses a pre-built whell. The pre-built PyTorch3D wheel is downloaded from my Dropbox (shared link). Another copy of the wheel is also stored in my Google Drive, and is located at: `/content/drive/MyDrive/Colab_wheels/`

## Load the CAD model file

We will load a CAD model (e.g., `ply` or `obj`) file and create a **Meshes** object. **Meshes** is a unique datastructure provided in PyTorch3D for working with **batches of meshes of different sizes**. It has several useful class methods which are used in the rendering pipeline.

#### ⚡ Install PyTorch3D from Wheel

PyTorch3D installation can take longer than 8-10 minutes when installed from source.

Here, **PyTorch3D is installed from a wheel** for a faster setup of about 2 minutes in Colab.

- If the installer instead tries to **build from source**, it means the wheel is outdated or missing.  
- In that case, you can **create your own wheel directly in Colab**, save it to **Google Drive** (or Dropbox), and reuse it later for faster installation.
- To create your own PyTorch3D wheel in Colab, follow the instructions in the cell after these installation cells.



In [ ]:
#@title Install PyTorch3D wheel from Google Drive, else fallback to Dropbox



# Try to install PyTorch3D from (pre-compiled) wheel stored in Google Drive or DropBox (or another server)

# ## Wheel created for A100 (It does not work on other instances, e.g., T4)
# !pip install /content/drive/MyDrive/pytorch3d_build/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl

# # This wheel is for T4 (Colab)
# !pip install /content/drive/MyDrive/pytorch3d_build/pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl



print("🚀 Installing PyTorch3D wheel based on selected Colab instance...")

if colab_instance == "A100":
    !pip install /content/drive/MyDrive/pytorch3d_build/A100/*.whl

elif colab_instance == "T4":
    !pip install /content/drive/MyDrive/pytorch3d_build/T4/*.whl

elif colab_instance == "L4":
    !pip install /content/drive/MyDrive/pytorch3d_build/L4/*.whl

else:
    raise ValueError("❌ No PyTorch3D wheel available for CPU mode!")







import os
import glob
import shutil
import zipfile
import subprocess
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
from IPython.display import display, HTML


# ============================================================
# Optional notebook message helper
# ============================================================
def show_message(message, title=None, level="info"):
    styles = {
        "info": {
            "bg": "linear-gradient(135deg, #f5f7fa, #e4ecf7)",
            "border": "#c9d7ea",
            "color": "#1f3b5b",
            "icon": "ℹ️"
        },
        "success": {
            "bg": "#ecfdf5",
            "border": "#a7f3d0",
            "color": "#065f46",
            "icon": "✅"
        },
        "warning": {
            "bg": "#fff7ed",
            "border": "#f59e0b",
            "color": "#7c2d12",
            "icon": "⚠️"
        },
        "error": {
            "bg": "#fef2f2",
            "border": "#ef4444",
            "color": "#7f1d1d",
            "icon": "❌"
        }
    }

    s = styles.get(level, styles["info"])
    title_html = f"<strong>{title}</strong><br>" if title else ""

    display(HTML(f"""
    <div style="
        padding: 12px 16px;
        margin: 8px 0;
        border-radius: 10px;
        background: {s['bg']};
        border-left: 6px solid {s['border']};
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
        font-family: Arial, sans-serif;
        font-size: 15px;
        color: {s['color']};
    ">
        {s['icon']} {title_html}
        {message}
    </div>
    """))


# ============================================================
# Main installer
# ============================================================
def install_pytorch3d_wheel(
    colab_instance,
    gdrive_base="/content/drive/MyDrive/pytorch3d_build",
    dropbox_folder_url=None,
    local_fallback_base="/content/pytorch3d_build_from_dropbox",
    dropbox_zip_path="/content/pytorch3d_build_dropbox.zip",
    verbose=True,
):
    """
    Install a PyTorch3D wheel for the selected Colab GPU type.

    Search order:
      1. Google Drive
      2. Dropbox shared folder fallback

    Expected directory structure:
      <gdrive_base>/A100/*.whl
      <gdrive_base>/T4/*.whl
      <gdrive_base>/L4/*.whl

    Same structure is assumed inside the Dropbox folder.
    """

    valid_instances = {"A100", "T4", "L4"}
    if colab_instance not in valid_instances:
        raise ValueError(
            f"❌ No PyTorch3D wheel available for '{colab_instance}'. "
            f"Supported values: {sorted(valid_instances)}"
        )

    def log(msg):
        if verbose:
            print(msg)

    def make_dropbox_download_url(url):
        parsed = urlparse(url)
        qs = parse_qs(parsed.query)
        qs["dl"] = ["1"]
        new_query = urlencode(qs, doseq=True)
        return urlunparse(parsed._replace(query=new_query))

    def find_wheels(base_dir, instance_name):
        patterns = [
            os.path.join(base_dir, instance_name, "*.whl"),
            os.path.join(base_dir, "**", instance_name, "*.whl"),
            os.path.join(base_dir, "**", "*.whl"),
        ]

        matches = []
        for pattern in patterns:
            matches.extend(glob.glob(pattern, recursive=True))

        # Deduplicate while preserving order
        unique = []
        seen = set()
        for path in matches:
            norm = os.path.normpath(path)
            if norm not in seen:
                unique.append(norm)
                seen.add(norm)

        # Prefer wheels explicitly under the requested instance folder
        preferred = [
            p for p in unique
            if f"/{instance_name}/" in p.replace("\\", "/")
        ]
        return preferred if preferred else unique

    def pip_install_wheel(wheel_path):
        log(f"📦 Installing wheel:\n{wheel_path}")
        result = subprocess.run(
            ["pip", "install", wheel_path],
            capture_output=True,
            text=True
        )

        if result.stdout.strip():
            print(result.stdout)

        if result.returncode != 0:
            if result.stderr.strip():
                print(result.stderr)
            raise RuntimeError(f"pip install failed for:\n{wheel_path}")

    def download_dropbox_folder_zip(url, zip_path):
        download_url = make_dropbox_download_url(url)
        log("⬇️ Downloading Dropbox folder as zip...")

        result = subprocess.run(
            ["wget", "-O", zip_path, download_url],
            capture_output=True,
            text=True
        )

        if result.returncode != 0:
            if result.stdout.strip():
                print(result.stdout)
            if result.stderr.strip():
                print(result.stderr)
            raise RuntimeError("Failed to download Dropbox folder zip.")

    def unzip_to(zip_path, out_dir):
        if os.path.exists(out_dir):
            shutil.rmtree(out_dir)
        os.makedirs(out_dir, exist_ok=True)

        log(f"🗂️ Extracting zip to {out_dir} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(out_dir)

    # --------------------------------------------------------
    # 1) Try Google Drive first
    # --------------------------------------------------------
    show_message(
        f"Looking for a PyTorch3D wheel for <b>{colab_instance}</b>.",
        title="PyTorch3D Installation",
        level="info"
    )

    gdrive_wheels = find_wheels(gdrive_base, colab_instance)

    if gdrive_wheels:
        wheel_path = gdrive_wheels[0]
        show_message(
            f"Found wheel in Google Drive:<br><code>{wheel_path}</code>",
            title="Wheel Found",
            level="success"
        )
        pip_install_wheel(wheel_path)
        show_message(
            f"PyTorch3D wheel installed successfully for <b>{colab_instance}</b> from Google Drive.",
            title="Installation Complete",
            level="success"
        )
        return wheel_path

    # --------------------------------------------------------
    # 2) Fallback to Dropbox
    # --------------------------------------------------------
    show_message(
        (
            f"No wheel was found in Google Drive for <b>{colab_instance}</b>.<br>"
            f"Falling back to the Dropbox copy."
        ),
        title="Google Drive Wheel Not Found",
        level="warning"
    )

    download_dropbox_folder_zip(dropbox_folder_url, dropbox_zip_path)
    unzip_to(dropbox_zip_path, local_fallback_base)

    dropbox_wheels = find_wheels(local_fallback_base, colab_instance)

    if not dropbox_wheels:
        show_message(
            (
                f"No <code>.whl</code> file was found for <b>{colab_instance}</b><br>"
                f"in either Google Drive or Dropbox."
            ),
            title="Installation Failed",
            level="error"
        )
        raise FileNotFoundError(
            f"No .whl file found for {colab_instance} in either Google Drive or Dropbox."
        )

    wheel_path = dropbox_wheels[0]
    show_message(
        f"Found wheel in Dropbox fallback:<br><code>{wheel_path}</code>",
        title="Wheel Found",
        level="success"
    )
    pip_install_wheel(wheel_path)
    show_message(
        f"PyTorch3D wheel installed successfully for <b>{colab_instance}</b> from Dropbox.",
        title="Installation Complete",
        level="success"
    )

    return wheel_path



# main
install_pytorch3d_wheel(colab_instance=colab_instance, dropbox_folder_url=dropbox_link)


In [ ]:
#@markdown ## ⚙️ Environment Setup and Dependency Initialization
#@markdown This cell prepares the execution environment and installs required dependencies for the PyTorch3D rendering workflow.

#@markdown ### 🔍 Platform Detection
#@markdown - Detects the current runtime environment (e.g., **Colab**, **Lightning AI**, local).
#@markdown - Configures paths and working directory (`/content/pytorch3d_rendering` if available).

#@markdown ### 📦 Module Import Handling
#@markdown - Attempts to import `installation_tools`:
#@markdown   - First from the current environment
#@markdown   - Falls back to the local repository if needed
#@markdown - Reloads the module to ensure latest updates are used

#@markdown ### 🛠️ Utility Functions
#@markdown - Defines helper functions for:
#@markdown   - Running shell commands (`run`)
#@markdown   - Installing packages via pip (`pip_install`)
#@markdown   - Optional conda installs (`conda_install`)

#@markdown ### ☁️ Optional Google Drive Mount
#@markdown - If enabled and running in Colab, mounts Google Drive for accessing external files

#@markdown ### ⚡ Platform-Specific Setup (Lightning AI)
#@markdown - Installs required system and Python dependencies
#@markdown - Applies compatibility fixes (e.g., `numpy<2.0`)
#@markdown - Installs additional libraries and optional `requirements.txt`

#@markdown ### 📚 Core Dependencies
#@markdown Installs required Python packages:
#@markdown - `trimesh`, `pyrender`, `opencv-python`
#@markdown - `matplotlib`, `pytorch-lightning`

#@markdown ### 🌐 External Utility Download
#@markdown - Downloads `plot_image_grid.py` from the official PyTorch3D repository if not already present

#@markdown ### ⬇️ Additional Tools
#@markdown - Installs `gdown` for downloading files from Google Drive

#@markdown ---
#@markdown ### ✅ Outcome
#@markdown After running this cell, the environment is fully configured and ready for:
#@markdown - Rendering
#@markdown - Visualization
#@markdown - PyTorch3D-based workflows

# # --- Config ---
# mount_gdrive = False

# # --- Imports ---
# import importlib, os, sys, shutil, subprocess, urllib.request, pathlib
# import installation_tools as install_tools
# importlib.reload(install_tools)

# --- Config ---
mount_gdrive = False

# --- Imports ---
import importlib, os, sys, shutil, subprocess, urllib.request, pathlib




if os.path.exists("/content/pytorch3d_rendering"):
    os.chdir("/content/pytorch3d_rendering")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

try:
    import installation_tools as install_tools
except ModuleNotFoundError:
    from pytorch3d_rendering import installation_tools as install_tools

importlib.reload(install_tools)




# --- Short helpers (no notebook magics) ---
def run(cmd, check=True):
    print("$", " ".join(cmd))
    try:
        subprocess.run(cmd, check=check)
    except subprocess.CalledProcessError as e:
        print(f"Command failed ({e.returncode}): {' '.join(cmd)}")
        if check:
            raise

def pip_install(*pkgs, extra=None, check=True):
    args = [sys.executable, "-m", "pip", "install"]
    if extra:
        args += extra
    args += list(pkgs)
    run(args, check=check)

def conda_available():
    return shutil.which("conda") is not None

def conda_install(*pkgs):
    if not conda_available():
        print("conda not available; skipping conda installs.")
        return
    # Use -c conda-forge channel and auto-yes
    run(["conda", "install", "-y", "-c", "conda-forge", *pkgs], check=False)

# --- Detect platform ---
pm = install_tools.PlatformManager()
platform, local_path = pm.platform, pm.local_path
print("Detected:", platform, local_path)

# --- Optional: Mount GDrive if on Colab ---
if mount_gdrive and platform == "Colab":
    pm.mount_gdrive()

# --- Lightning AI specific environment tweaks ---
if platform == "LightningAI":
    # conda piece (if conda exists in the image)
    conda_install("libstdcxx-ng=13")
    # pip pins / extras
    pip_install("numpy<2.0", check=False)
    pip_install("scikit-image", "gradio", "moviepy", "plotly", check=False)
    # If requirements.txt exists in CWD, install it
    if os.path.exists("requirements.txt"):
        pip_install("-r", "requirements.txt")


# --- Extra libraries (quiet-ish) ---
# Original line had: trimesh pyrender opencv-python matplotlib pytorch-lightning
pip_install("trimesh", "pyrender", "opencv-python", "matplotlib", "pytorch-lightning", check=False)

# --- Download plot_image_grid.py if missing ---
filename = "plot_image_grid.py"
url = "https://raw.githubusercontent.com/facebookresearch/pytorch3d/main/docs/tutorials/utils/plot_image_grid.py"
if not os.path.exists(filename):
    print(f"Downloading {filename} ...")
    try:
        urllib.request.urlretrieve(url, filename)
        print("Saved to", pathlib.Path(filename).resolve())
    except Exception as e:
        print("Download failed:", e)

# --- gdown ---
pip_install("gdown", extra=["--quiet"], check=False)
print("✅ Setup complete.")


**Install and import colorama module (color printing)**

In [ ]:
!pip install colorama
from colorama import Fore, Back, Style, init

# ---------- pretty print helpers ----------
RESET="\033[0m"; BOLD="\033[1m"
C={"ok":"\033[1;32m","info":"\033[1;36m","step":"\033[1;35m","warn":"\033[1;33m"}
CYAN  = "\033[1;36m"; GREEN = "\033[1;32m"; YELLOW = "\033[1;33m"


def say(kind,msg): print(f"{C[kind]}{msg}{RESET}")
torch.set_printoptions(precision=4, sci_mode=False)
np.set_printoptions(precision=4, suppress=True)




#### 🛠️ (Optional) Build Your Own PyTorch3D Wheel

If the pre-built PyTorch3D wheel does not match your setup, you can **build PyTorch3D from source** and save the wheel to Google Drive.  
This way, you only build once and reuse the `.whl` file in future Colab sessions.

To rebuild the PyTorch3D wheel run the notebook: `pytorch3d_wheel_builder.ipynb` in this project's repository.



#### PyTorch3D imports
The following cell require PyTorch3D. Ensure it is executed after PyTorch3D is installed.

In [ ]:
# # ---------------------------- IMPORTS -----------------------------------------
# PyTorch3D — IO & data structures
from pytorch3d.io import load_obj, load_ply, load_objs_as_meshes
from pytorch3d.structures import Meshes

# PyTorch3D — transforms
from pytorch3d.transforms import Rotate, Translate

# PyTorch3D — rendering
from pytorch3d.renderer import (
    FoVPerspectiveCameras,
    PerspectiveCameras,
    look_at_view_transform,
    look_at_rotation,
    camera_position_from_spherical_angles,
    RasterizationSettings,
    MeshRenderer,
    MeshRasterizer,
    BlendParams,
    SoftSilhouetteShader,
    SoftPhongShader,
    HardPhongShader,
    PointLights,
    DirectionalLights,
    Materials,
    TexturesUV,
    TexturesVertex,
)
from pytorch3d.renderer.cameras import CamerasBase

# PyTorch3D — visualization helpers (optional)
from pytorch3d.vis.plotly_vis import AxisArgs, plot_batch_individually, plot_scene
from pytorch3d.vis.texture_vis import texturesuv_image_matplotlib

from pytorch3d.utils import cameras_from_opencv_projection


# Project utils path (adjust as needed)
sys.path.append(os.path.abspath(''))
# ------------------------------------------------------------------------------


In [ ]:
# # ---------------------------- IMPORTS -----------------------------------------
# # Stdlib
# import os
# import sys
# import math
# import shutil
# from pathlib import Path
# from typing import Optional, Tuple, Literal, Dict, Any

# # Third-party
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import cv2
# import imageio
# import requests
# import matplotlib.pyplot as plt
# import matplotlib.cm as cm
# from tqdm.notebook import tqdm
# from skimage import img_as_ubyte

# # PyTorch3D — IO & data structures
# from pytorch3d.io import load_obj, load_ply, load_objs_as_meshes
# from pytorch3d.structures import Meshes

# # PyTorch3D — transforms
# from pytorch3d.transforms import Rotate, Translate

# # PyTorch3D — rendering
# from pytorch3d.renderer import (
#     FoVPerspectiveCameras,
#     PerspectiveCameras,
#     look_at_view_transform,
#     look_at_rotation,
#     camera_position_from_spherical_angles,
#     RasterizationSettings,
#     MeshRenderer,
#     MeshRasterizer,
#     BlendParams,
#     SoftSilhouetteShader,
#     SoftPhongShader,
#     HardPhongShader,
#     PointLights,
#     DirectionalLights,
#     Materials,
#     TexturesUV,
#     TexturesVertex,
# )
# from pytorch3d.renderer.cameras import CamerasBase

# # PyTorch3D — visualization helpers (optional)
# from pytorch3d.vis.plotly_vis import AxisArgs, plot_batch_individually, plot_scene
# from pytorch3d.vis.texture_vis import texturesuv_image_matplotlib

# # Project utils path (adjust as needed)
# sys.path.append(os.path.abspath(''))
# # ------------------------------------------------------------------------------


If using **Google Colab**, fetch the utils file for plotting image grids:

In [ ]:
!wget https://raw.githubusercontent.com/facebookresearch/pytorch3d/main/docs/tutorials/utils/plot_image_grid.py
from plot_image_grid import image_grid

In [ ]:
# ---------- pretty print helpers ----------
RESET="\033[0m"; BOLD="\033[1m"
C={"ok":"\033[1;32m","info":"\033[1;36m","step":"\033[1;35m","warn":"\033[1;33m"}
CYAN  = "\033[1;36m"; GREEN = "\033[1;32m"; YELLOW = "\033[1;33m"


def say(kind,msg): print(f"{C[kind]}{msg}{RESET}")
torch.set_printoptions(precision=4, sci_mode=False)
np.set_printoptions(precision=4, suppress=True)


# PyTorch3D Coordinate systems: World and Camera

This example shows a plot of the PyTorch3D coordinate system and the camera coordinate system. The camera is shown at two different positions along the z-axis, i.e.: elev=azim=0 and elev=0/azim=180.

## Helper functions

In [ ]:
import tools_pytorch3d_coordsystems_demo as myp3dtools
import tools_image_processing as myimgtools



## **Example**: Location of cameras and the world coordinate systems

In [ ]:
# --- Demo: azim = 0 vs 180 ---
device = torch.device("cpu")
dist, elev = 3.0, 0.0

fig = plt.figure(figsize=(12,6))
for i, azim in enumerate([0.0, 180.0]):
    R, T = look_at_view_transform(dist=dist, elev=elev, azim=azim, device=device)
    ax = fig.add_subplot(1,2,i+1, projection='3d')
    myp3dtools.plot_world_axes(ax, length=1.0)
    myp3dtools.plot_camera_axes_and_ray(R, T, ax, label=f"azim={azim:.0f}")

    # Nice bounds / labels
    s = dist + 0.5
    ax.set_xlim(-s, s); ax.set_ylim(-s, s); ax.set_zlim(-s, s)
    ax.set_xlabel("X"); ax.set_ylabel("Y"); ax.set_zlabel("Z")
    ax.set_title(f"Camera at elev=0, azim={azim:.0f}")

plt.tight_layout()
plt.show()


## **Example**: Azimuth (orbital) path plot of the PyTorch3D camera

In [ ]:

# Run it
myp3dtools.plot_orbit_with_cam_dirs(dist=3.0, elev=0.0, step_curve=5, step_arrows=45)


# Rendering textured meshes

## Load a mesh and texture file

Load an `.obj` file and its associated `.mtl` file and create a **Textures** and **Meshes** object.

**Meshes** is a unique datastructure provided in PyTorch3D for working with batches of meshes of different sizes.

**TexturesUV** is an auxiliary datastructure for storing vertex uv and texture maps for meshes.

**Meshes** has several class methods which are used throughout the rendering pipeline.

If running this notebook using **Google Colab**, run the following cell to fetch the mesh obj and texture files and save it at the path `data/cow_mesh`:
If running locally, the data is already available at the correct path.

### Download mesh files

In [ ]:
# Example usage
myp3dtools.download_cow_mesh()

### Load cad model into a Pytorch3D mesh

In [ ]:
# Setup
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")

# Set paths
DATA_DIR = "./data"
obj_filename = os.path.join(DATA_DIR, "cow_mesh/cow.obj")

# Load obj file
mesh = load_objs_as_meshes([obj_filename], device=device)

### Let's visualize the texture map

PyTorch3D has a built-in way to view the texture map with matplotlib along with the points on the map corresponding to vertices. There is also a method, texturesuv_image_PIL, to get a similar image which can be saved to a file.

In [ ]:
plt.figure(figsize=(7,7))
texture_image=mesh.textures.maps_padded()
plt.imshow(texture_image.squeeze().cpu().numpy())
plt.axis("off");

In [ ]:
plt.figure(figsize=(7,7))
texturesuv_image_matplotlib(mesh.textures, subsample=None)
plt.axis("off");

## Initialize the camera (and get your bearings)

* **Default view.** `look_at_view_transform(dist, elev=0, azim=0)` places the camera on the **+Z** axis at `(0, 0, dist)`, looking at the origin.
  The returned rotation is:

  $$
  R=\mathrm{diag}(-1,\;1,\;-1),
  $$

  which means (rows of $R$ = camera axes in world coords):

  * $+X_{\text{cam}}$ points along $-X_{\text{world}}$
  * $+Y_{\text{cam}}$ points along $+Y_{\text{world}}$
  * $+Z_{\text{cam}}$ points along $-Z_{\text{world}}$

  Practically: a small **+X\_world** step projects **to the right** in the image; **+Y\_world** projects **up** (note: image $v$ increases downward).

* **Opposite viewpoint.** To look from the other side (i.e., have the view direction along **+Z\_world**), set `azim=180` (with the same `elev=0`).
  This moves the camera to `(0, 0, -dist)` while still looking at the origin.

* **About the cow mesh.** The canonical PyTorch3D cow’s **front** faces **Z\_world**.

  * `azim=0` → rear view
  * `azim=180` → front view


> Tip: If you print `R` and take its **rows** as vectors, you’re reading the camera’s +X/+Y/+Z axes expressed in the **world** frame—handy for sanity checks.


## Create a renderer
Here, I am creating a general renderer (Soft Phong). We can later change its settings such as lights, camera.

### What requires a rebuild?

- Change **image size**, **faces_per_pixel**, **blur_radius**, **bin_size/max_faces_per_bin** → recreate rasterizer/renderer (or replace `renderer.rasterizer.raster_settings` with a new one).
- Change **shader type** or **blend params** → recreate/replace shader.

Everything else (new `R,T`, different intrinsics, per-frame light positions, materials) can be passed each call.


In [ ]:
# Image size
W = 256
H = 256

# Create general (Soft) Phong renderer
phong_renderer = myp3dtools.make_phong_renderer(W, H, device)

## Camera position: Facing the front of the *cow*

In [ ]:
#          Camera pose in spherical coordinates
#==============================================================================
distance, elev, azim = 3, 0.0, 180.0   # front
#==============================================================================

# Pretty printing
myp3dtools.print_spherical_coords(distance, elev, azim)

# Get the camera pose (Row-major storage)
R, T = look_at_view_transform(distance, elev, azim)

# Pretty print camera information
myp3dtools.print_camera_pose_matrices(R, T, "PyTorch3D Camera")

# Create camera(s)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

# Get camera center in world coordinates (handles batches)
light_loc = cameras.get_camera_center()          # shape: (N, 3)

# Create a Lights module at the camera center
lights = PointLights(device=device, location=cameras.get_camera_center())

# Set the renderer state as needed for the image(s)
myp3dtools.set_renderer_state(phong_renderer, cameras=cameras, lights=lights)

# Render images
images = phong_renderer(mesh)

# Display image
plt.figure(figsize=(5, 5))
plt.imshow(images[0, ..., :3].cpu().numpy())
plt.axis("off");



## Camera position: Facing the back of the *cow*

In [ ]:
#          Camera pose in spherical coordinates
#==============================================================================
distance, elev, azim = 3, 0.0, 0.0   # front
#==============================================================================

# Pretty printing
myp3dtools.print_spherical_coords(distance, elev, azim)

# Get the camera pose (Row-major storage)
R, T = look_at_view_transform(distance, elev, azim)

# Pretty print camera information
myp3dtools.print_camera_pose_matrices(R, T, "PyTorch3D Camera")

# Create camera(s)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

# Get camera center in world coordinates (handles batches)
light_loc = cameras.get_camera_center()          # shape: (N, 3)

# Create a Lights module at the camera center
lights = PointLights(device=device, location=cameras.get_camera_center())

# Set the renderer state as needed for the image(s)
myp3dtools.set_renderer_state(phong_renderer, cameras=cameras, lights=lights)

# Render images
images = phong_renderer(mesh)

# Display image
plt.figure(figsize=(5, 5))
plt.imshow(images[0, ..., :3].cpu().numpy())
plt.axis("off");



## Rotate the object
Rotate the object by changing the elevation and azimuth angles


In [ ]:

#          Camera pose in spherical coordinates
#==============================================================================
distance, elev, azim = 3, 30.0, 170.0   # front
#==============================================================================

# Pretty printing
myp3dtools.print_spherical_coords(distance, elev, azim)

# Get the camera pose (Row-major storage)
R, T = look_at_view_transform(distance, elev, azim)

# Pretty print camera information
myp3dtools.print_camera_pose_matrices(R, T, "PyTorch3D Camera")

# Create camera(s)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

# Get camera center in world coordinates (handles batches)
light_loc = cameras.get_camera_center()          # shape: (N, 3)

# Create a Lights module at the camera center
lights = PointLights(device=device, location=cameras.get_camera_center())

# Set the renderer state as needed for the image(s)
myp3dtools.set_renderer_state(phong_renderer, cameras=cameras, lights=lights)

# Render images
images = phong_renderer(mesh)

rgb_rotated = images[0, ..., :3].cpu().numpy()

# Display image
plt.figure(figsize=(5, 5))
plt.imshow(rgb_rotated)
plt.axis("off");


## Show coordinate system
Here, we plot the world coordinate system (PyTorch3D)

In [ ]:
myp3dtools.overlay_axes_p3d(rgb_rotated, cameras, 256, 256,
                 world_origin=(0,0,0), axis_len=0.5,
                 draw_world_axes=True, draw_camera_axes=False,
                 cam_axis_len=0.5,
                 title="Axes overlay")

###  Overlay quadrant sample points (PyTorch3D)

In [ ]:
# ---- Config: sample points around the world origin on the z=0 plane ----------
L = 0.25  # distance from origin (world units)
samples_world = np.array([
    [ +L, +L, 0.0],  # (+,+)
    [ -L, +L, 0.0],  # (-,+)
    [ -L, -L, 0.0],  # (-,-)
    [ +L, -L, 0.0],  # (+,-)
    [  0,   0, 0.0], # origin
], dtype=np.float32)



In [ ]:
# === Overlay quadrant sample points (PyTorch3D) ===============================
import torch, numpy as np, matplotlib.pyplot as plt

# ---- Project to screen --------------------------------------------------------
device = cameras.R.device
dtype  = cameras.R.dtype
imgsz  = torch.tensor([[H, W]], device=device)

Xw = torch.from_numpy(samples_world).to(device=device, dtype=dtype)[None]  # (1,N,3)

# Pixel projection (v increases downward)
uvz = cameras.transform_points_screen(Xw, image_size=imgsz)[0]  # (N,3)
uv  = uvz[:, :2].detach().cpu().numpy()

# Also compute camera coords to show sign mapping
R = cameras.R[0]            # (3,3)
T = cameras.T[0]            # (3,)
Xc = (Xw[0] @ R.T) + T      # (N,3) row-vector convention

# ---- Draw --------------------------------------------------------------------
plt.figure(figsize=(6,6))
plt.imshow(rgb_rotated)
plt.axis("off")

for i, (pt_w, (u,v), pt_c) in enumerate(zip(samples_world, uv, Xc.detach().cpu().numpy())):
    xw, yw, zw = pt_w
    xc, yc, zc = pt_c
    col = myp3dtools.color_for(xw, yw)
    m   = myp3dtools.marker_for(xw, yw)

    # scatter the point
    plt.scatter([u], [v], s=60 if i>0 else 80, c=col, marker=m, edgecolors='k' if i>0 else 'k', linewidths=1.0, zorder=3)

    # label with world sign and (optional) camera sign
    label = f"({myp3dtools.sign2(xw)}x_w, {myp3dtools.sign2(yw)}y_w)\n({myp3dtools.sign2(xc)}x_c, {myp3dtools.sign2(yc)}y_c)"
    # offset text a bit so labels don't overlap markers
    plt.text(u+4, v-6, label, color=col if i>0 else "black",
             fontsize=8, weight="bold", bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=2))

# Crosshair at principal point (for reference)
cx = float(cameras.principal_point[0,0]) if hasattr(cameras, "principal_point") else W/2
cy = float(cameras.principal_point[0,1]) if hasattr(cameras, "principal_point") else H/2
plt.plot([cx-10, cx+10], [cy, cy], color="white", lw=1)
plt.plot([cx, cx], [cy-10, cy+10], color="white", lw=1)
plt.title("Quadrant samples: world-sign (top) / camera-sign (bottom)")
plt.show()

# ---- Console summary ----------------------------------------------------------
print("Index | World (x,y,z)         | Camera (x,y,z)         | Pixel (u,v)")
for i, (pt_w, pt_c, (u,v)) in enumerate(zip(samples_world, Xc.detach().cpu().numpy(), uv)):
    print(f"{i:5d} | {pt_w[0]:+7.3f} {pt_w[1]:+7.3f} {pt_w[2]:+7.3f} | "
          f"{pt_c[0]:+7.3f} {pt_c[1]:+7.3f} {pt_c[2]:+7.3f} | "
          f"{u:7.2f} {v:7.2f}")


In [ ]:
# === Overlay quadrant sample points with indices ==============================
import torch, numpy as np, matplotlib.pyplot as plt

# ---- Project to screen --------------------------------------------------------
device = cameras.R.device
dtype  = cameras.R.dtype
imgsz  = torch.tensor([[H, W]], device=device)

Xw = torch.from_numpy(samples_world).to(device=device, dtype=dtype)[None]  # (1,N,3)
uvz = cameras.transform_points_screen(Xw, image_size=imgsz)[0]             # (N,3)
uv  = uvz[:, :2].detach().cpu().numpy()

# Camera-frame coordinates (row-vector convention)
R = cameras.R[0]; T = cameras.T[0]
Xc = (Xw[0] @ R.T) + T    # (N,3)

# ---- Draw --------------------------------------------------------------------
plt.figure(figsize=(6,6))
plt.imshow(rgb_rotated)


plt.axis("off")

for i, (pt_w, (u,v), pt_c) in enumerate(zip(samples_world, uv, Xc.detach().cpu().numpy())):
    xw, yw, zw = pt_w
    xc, yc, zc = pt_c
    col = myp3dtools.color_for(xw, yw)

    # scatter the point
    plt.scatter([u], [v], s=70, c=col, marker="o", edgecolors="k", linewidths=1.0, zorder=3)

    # index label next to the dot
    plt.text(u+5, v-5, f"{i}", fontsize=10, weight="bold",
             bbox=dict(facecolor="white", alpha=0.85, edgecolor="black", linewidth=0.5, pad=1.5))

# Crosshair at principal point (for reference)
cx = float(cameras.principal_point[0,0]) if hasattr(cameras, "principal_point") else W/2
cy = float(cameras.principal_point[0,1]) if hasattr(cameras, "principal_point") else H/2
plt.plot([cx-10, cx+10], [cy, cy], color="white", lw=1)
plt.plot([cx, cx], [cy-10, cy+10], color="white", lw=1)
plt.title("Quadrant samples with indices")
plt.show()

# ---- Console summary (indexed) -----------------------------------------------
print("Index | World (x,y,z)         | Camera (x,y,z)         | Pixel (u,v)")
for i, (pt_w, pt_c, (u,v)) in enumerate(zip(samples_world, Xc.detach().cpu().numpy(), uv)):
    print(f"{i:5d} | {pt_w[0]:+7.3f} {pt_w[1]:+7.3f} {pt_w[2]:+7.3f} | "
          f"{pt_c[0]:+7.3f} {pt_c[1]:+7.3f} {pt_c[2]:+7.3f} | "
          f"{u:7.2f} {v:7.2f}")


# Rendering images using PyTorch3D and OpenCV camera poses (R,T)



## Example 1:
This example shows how to render an image using a PyTorch3D  camera created from an OpenCV camera intrinsics using `PerspectiveCamera()`. The same view is also rendered using a simple projection function that uses opencv coordinate system. The conversion from Pytorch3D (R_p3d, t_p3d) to OpenCV (R_cv, t_cv) is handled by the PyTorch3D helper function `opencv_from_cameras_projection()`.

### Perspective PyTorch3D camera from K and R,T in PyTorch3D coordinate system

In [ ]:
# Define camera and image settings using intrinsics in pixels (OpenCV-style)
H, W = 256, 256
image_size = torch.tensor([[H, W]], dtype=torch.float32, device=device)
fx = fy = 200.0
cx, cy = W/2, H/2

In [ ]:
import torch
from pytorch3d.renderer import PerspectiveCameras
from pytorch3d.utils import opencv_from_cameras_projection, cameras_from_opencv_projection

# --- PyTorch3D + OpenCV: render the same pose using an EXISTING mesh ----------
# Requirements: `mesh` is a PyTorch3D Meshes with a single item (batch=1).
# If it has no textures, a flat vertex color is applied for PyTorch3D shading.

import numpy as np
import torch, cv2
import matplotlib.pyplot as plt

from pytorch3d.renderer import (
    MeshRenderer, MeshRasterizer, SoftPhongShader, RasterizationSettings,
    PointLights, PerspectiveCameras, Materials
)
from pytorch3d.structures import Meshes
from pytorch3d.renderer import TexturesVertex


#------------------------------------------------------------------------------
# Pose in PyTorch3D (world -> view)
dist=3
elev=10
azim=180
R_p3d, T_p3d = look_at_view_transform(dist=dist, elev=elev, azim=azim, device=device)

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ----------------------------- Assume existing mesh ---------------------------
# `mesh` must exist: a PyTorch3D Meshes (batch=1). Example:
# assert isinstance(mesh, Meshes), "`mesh` must be a PyTorch3D Meshes."


# -- PyTorch3D camera -------------------------------
cams = PerspectiveCameras(
    focal_length=((fx, fy),),
    principal_point=((cx, cy),),
    image_size=((H, W),),
    R=R_p3d, T=T_p3d,
    in_ndc=False,
    device=device
)

# Pretty printing
myp3dtools.print_spherical_coords(distance, elev, azim)

# Pretty print camera information
myp3dtools.print_camera_pose_matrices(R_p3d, T_p3d, "*** PyTorch3D Camera ***")

# Get camera center in world coordinates (handles batches)
light_loc = cams.get_camera_center()          # shape: (N, 3)

# Create a Lights module at the camera center
lights = PointLights(device=device, location=cams.get_camera_center())

# Set the renderer state as needed for the image(s)
myp3dtools.set_renderer_state(phong_renderer, cameras=cams, lights=lights)

# Render images
images = phong_renderer(mesh)

# Make tensor into rgb
p3d_img = images[0, ..., :3].cpu().numpy()


# -- Show results ---------------------------------
myp3dtools.overlay_axes_p3d(p3d_img, cams, 256, 256,
                 world_origin=(0,0,0), axis_len=0.5,
                 draw_world_axes=True, draw_camera_axes=False,
                 cam_axis_len=0.5,
                 title="PyTorch3D camera")

### Using PyTorch3D helper function (R_p3d, T_p3d) to (R_cv,T_cv) -> OpenCV rendering

In [ ]:
# Image size as torch tensor
image_size=torch.tensor([[H, W]], dtype=torch.float32, device=device)

# Get R, T in OpenCV coordinates (as torch tensors) - PyTorch3D helper
R_cv, t_cv, _ = opencv_from_cameras_projection(cams, image_size)

# Pretty print camera information
myp3dtools.print_camera_pose_matrices(R_cv, t_cv, "*** OpenCV Camera ***")

# Print transpose of tensor rotation matrix
light_loc = myp3dtools.camera_center_from_RT_openCV(R_cv, t_cv)
print(f"\nCamera center (OpenCV): {light_loc}\n")

In [ ]:
# Build K and OpenCV extrinsics corresponding to the same physical pose
# The conversion pytorch3d-to-opencv needs K as a tensor
# Helper cameras_from_opencv_projection expects K as a tensor
K = torch.tensor([[[fx, 0., cx],
                   [0., fy, cy],
                   [0.,  0., 1.]]], device=device)

# Convert K to numpy (simple matrix)
K_cv = K.squeeze().cpu().numpy()

# Render image of the mesh using opencv lines, polygons
img_cv_rgb = myp3dtools.render_image_with_opencv_lines_opencv_coordinates(mesh, R_cv, t_cv, K_cv, W, H)


#------------------------------- Show results ---------------------------------
myp3dtools.overlay_axes_p3d(img_cv_rgb, cams, 256, 256,
                 world_origin=(0,0,0), axis_len=0.5,
                 draw_world_axes=True, draw_camera_axes=False,
                 cam_axis_len=0.5,
                 title="OpenCV camera")


## Example 2:  
Rendering in PyTorch3D from a camera created from OpenCV `(R_cv, t_cv)` using the PyTorch3D helper function `cameras_from_opencv_projection()`.

In [ ]:

# ---------- Build camera from OpenCV K,R,t ----------
say("step", "\n🎯 Building PerspectiveCameras from OpenCV (K, R_cv, t_cv)")
# image_size is (N,2) tensor or list-like (H,W); device must match your setup
cameras_new = cameras_from_opencv_projection(
    R=R_cv, tvec=t_cv, camera_matrix=K, image_size=image_size)

say("ok", "📸 PyTorch3D camera created from OpenCV intrinsics passed to helper function")
print(f"{BOLD}K (OpenCV):{RESET}\n{K}")
print(f"{BOLD}image_size:{RESET} {cameras_new.image_size}")
print(f"{BOLD}focal_length (pixels):{RESET} {cameras_new.focal_length}")
print(f"{BOLD}principal_point (pixels):{RESET} {cameras_new.principal_point}")


# Pretty print camera pose information
myp3dtools.print_camera_pose_matrices(cameras_new.R, cameras_new.T, "*** PyTorch3D Camera ***")

# Get camera center in world coordinates (handles batches)
light_loc = cameras_new.get_camera_center()          # shape: (N, 3)

# Create a Lights module at the camera center
lights = PointLights(device=device, location=cameras_new.get_camera_center())

# Set the renderer state as needed for the image(s)
myp3dtools.set_renderer_state(phong_renderer, cameras=cameras_new, lights=lights)

# Render images
images = phong_renderer(mesh)

rgb_p3d = images[0, ..., :3].cpu().numpy()




# ------------------------------- Show results ---------------------------------
myp3dtools.overlay_axes_p3d(rgb_p3d, cameras_new, 256, 256,
                 world_origin=(0,0,0), axis_len=0.5,
                 draw_world_axes=True, draw_camera_axes=False,
                 cam_axis_len=0.5,
                 title="PyTorch3D camera")



## Example 3:  Depth map

In [ ]:
def depth_to_rgb(
    depth: torch.Tensor,
    cmap: str = "viridis",
    bg_mode: Literal["black","white","transparent"] = "black"
) -> np.ndarray:
    """
    Convert a PyTorch3D depth map into a visualization image.
    Automatically treats depth = -1 as invalid background.

    Args:
        depth: (H,W) torch.Tensor of depth values (float).
        cmap:  Matplotlib colormap name (default: "viridis").
        bg_mode:
            "black"       -> background stays black (default)
            "white"       -> background set to white
            "transparent" -> return RGBA image with alpha=0 for background

    Returns:
        img: (H,W,3) uint8 RGB or (H,W,4) uint8 RGBA if bg_mode="transparent"
    """
    if not isinstance(depth, torch.Tensor):
        raise ValueError("depth must be a torch.Tensor")

    depth_np = depth.detach().cpu().numpy().astype(np.float32)

    # Mask out invalid regions (zbuf = -1 means background in PyTorch3D)
    valid_mask = depth_np > 0

    if np.any(valid_mask):
        dmin, dmax = depth_np[valid_mask].min(), depth_np[valid_mask].max()
        depth_norm = np.zeros_like(depth_np, dtype=np.float32)
        depth_norm[valid_mask] = (depth_np[valid_mask] - dmin) / (dmax - dmin + 1e-8)
    else:
        depth_norm = np.zeros_like(depth_np, dtype=np.float32)

    cmap_func = cm.get_cmap(cmap)
    rgba = cmap_func(depth_norm)  # (H,W,4) floats in [0,1]

    if bg_mode == "transparent":
        rgba[..., 3] = 0.0          # alpha=0 where background
        rgba[valid_mask, 3] = 1.0   # alpha=1 where valid
        img = (rgba * 255).astype(np.uint8)  # RGBA
    else:
        rgb = (rgba[..., :3] * 255).astype(np.uint8)
        if bg_mode == "white":
            rgb[~valid_mask] = [255,255,255]
        elif bg_mode == "black":
            rgb[~valid_mask] = [0,0,0]
        img = rgb

    return img


In [ ]:
fragments = phong_renderer.rasterizer(mesh)
depth_map = fragments.zbuf[0, ..., 0].detach()                     # (H,W) torch.float32

# Depth visualization (treat -1 as invalid)
vis1 = depth_to_rgb(depth_map, cmap="hot", bg_mode="white")

# ------------------------------- Show results ---------------------------------
myp3dtools.overlay_axes_p3d(vis1, cameras_new, 256, 256,
                 world_origin=(0,0,0), axis_len=0.5,
                 draw_world_axes=True, draw_camera_axes=False,
                 cam_axis_len=0.5,
                 title="PyTorch3D camera")


# Depth visualization (treat -1 as invalid)
vis2_rgba = depth_to_rgb(depth_map, cmap="hot", bg_mode="transparent")

# ------------------------------- Show results ---------------------------------
myp3dtools.overlay_axes_p3d(vis2_rgba, cameras_new, 256, 256,
                 world_origin=(0,0,0), axis_len=0.5,
                 draw_world_axes=True, draw_camera_axes=False,
                 cam_axis_len=0.5,
                 title="PyTorch3D camera")


# Batched Rendering

One of the core design choices of the PyTorch3D API is to support **batched inputs for all components**.
The renderer and associated components can take batched inputs and **render a batch of output images in one forward pass**. We will now use this feature to render the mesh from many different viewpoints.


## Create a renderer

A renderer in PyTorch3D is composed of a **rasterizer** and a **shader** which each have a number of subcomponents such as a **camera** (orthographic/perspective). Here we initialize some of these components and use default values for the rest.

In this example we will first create a **renderer** which uses a **perspective camera**, a **point light** and applies **Phong shading**. Then we learn how to vary different components using the modular API.  

In [ ]:
# Initialize a camera.
# With world coordinates +Y up, +X left and +Z in, the front of the cow is facing the -Z direction.
# So we move the camera by 180 in the azimuth direction so it is facing the front of the cow.
# R, T = look_at_view_transform(2.7, 0, 180)
# cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

# Define the settings for rasterization and shading. Here we set the output image to be of size
# 512x512. As we are rendering images for visualization purposes only we will set faces_per_pixel=1
# and blur_radius=0.0. We also set bin_size and max_faces_per_bin to None which ensure that
# the faster coarse-to-fine rasterization method is used. Refer to rasterize_meshes.py for
# explanations of these parameters. Refer to docs/notes/renderer.md for an explanation of
# the difference between naive and coarse-to-fine rasterization.
raster_settings = RasterizationSettings(
    image_size=256,
    blur_radius=0.0,
    faces_per_pixel=1,
)

# Place a point light in front of the object. As mentioned above, the front of the cow is facing the
# -z direction.
lights = PointLights(device=device, location=[[0.0, 0.0, -3.0]])

# Create a Phong renderer by composing a rasterizer and a shader. The textured Phong shader will
# interpolate the texture uv coordinates for each vertex, sample from a texture image and
# apply the Phong lighting model
renderer = MeshRenderer(
    rasterizer=MeshRasterizer(
        raster_settings=raster_settings
    ),
    shader=SoftPhongShader(
        device=device,
        cameras=cameras,
    )
)

In [ ]:
# Set batch size - this is the number of different viewpoints from which we want to render the mesh.
batch_size = 20

# Create a batch of meshes by repeating the cow mesh and associated textures.
# Meshes has a useful `extend` method which allows us do this very easily.
# This also extends the textures.
meshes = mesh.extend(batch_size)

# Get a batch of viewing angles.
elev = torch.linspace(0, 180, batch_size)
azim = torch.linspace(-180, 180, batch_size)

# All the cameras helper methods support mixed type inputs and broadcasting. So we can
# view the camera from the same distance and specify dist=2.7 as a float,
# and then specify elevation and azimuth angles for each viewpoint as tensors.
R, T = look_at_view_transform(dist=2.7, elev=elev, azim=azim)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

# Move the light back in front of the cow which is facing the -z direction.
lights.location = torch.tensor([[0.0, 0.0, -3.0]], device=device)

# We can pass arbitrary keyword arguments to the rasterizer/shader via the renderer
# so the renderer does not need to be reinitialized if any of the settings change.
images = renderer(meshes, cameras=cameras, lights=lights)

# Show images as a grid
image_grid(images.cpu().numpy(), rows=4, cols=5, rgb=True)

# Creating masks, transparent images, contours, and blending images

### Another function that displays a wireframe

In [ ]:
import matplotlib.pyplot as plt
from pytorch3d.io import load_objs_as_meshes
from pytorch3d.renderer.cameras import PerspectiveCameras

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

wireframe_image = myimgtools.ImageProcessor.mesh_wireframe_image(
    mesh, cameras_new, (H, W),
    edge_mode="all", feature_deg=60.0,
    line_rgb=(200, 0, 0), line_thickness=1
)

plt.figure(figsize=(7,5))
plt.imshow(wireframe_image); plt.axis("off"); plt.title("Wireframe (RGB)")
plt.show()


### Creating masks and contours

In [ ]:
# Create some useful images to superimpose on the reference image during matching
contour_img, mask, contour_only = myimgtools.ImageProcessor.contour_from_nonwhite_rgb(
    rgb_p3d, draw_color=(255, 0, 0), return_masks = True)

plt.figure(figsize=(4,4))
plt.imshow(rgb_p3d); plt.axis("off"); plt.title("Input RGB image")
plt.show()


# Create a single row with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(15,5))

# First image
axes[0].imshow(contour_img)
axes[0].axis("off")
axes[0].set_title("RGB + Contour")

# Second image
axes[1].imshow(mask, cmap="gray")
axes[1].axis("off")
axes[1].set_title("Mask")

# Third image
axes[2].imshow(contour_only)
axes[2].axis("off")
axes[2].set_title("Contour Only")

plt.tight_layout()
plt.show()


### Depth map display: white background and transparent (alpha)

Here, the background can be either white or transparent. The object region is not transparent.

In [ ]:

# depth is a (H,W) torch tensor from PyTorch3D fragments.zbuf
depth_rgb_white_bg = depth_to_rgb(depth_map, cmap="plasma", bg_mode="white")
depth_rgba_transparent_bg = depth_to_rgb(depth_map, cmap="plasma", bg_mode="transparent")

fig, axes = plt.subplots(1, 2, figsize=(12,5))

# Depth with white background
axes[0].imshow(depth_rgb_white_bg)
axes[0].axis("off")
axes[0].set_title("Depth visualization (white background)")

# Depth with transparent background
axes[1].imshow(depth_rgba_transparent_bg)
axes[1].axis("off")
axes[1].set_title("Depth with transparent background")

plt.tight_layout()
plt.show()

### Display semi-transparent overlay
In this visualization, the object region is transparent but not the contour line. The line remains opaque. This visualization is good to show alignment.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assume you already have your RGBA result
contour_rgba = myimgtools.ImageProcessor.contour_from_nonwhite_rgb_semi_transparent(
    rgb_p3d,
    draw_color=(255,0,0),
    thickness=2,
    return_rgba=True,
    alpha_value=0.4
)

# --- Create a checkerboard background ---
def checkerboard_2x2(h, w):
    """Generate a 2x2 checkerboard (4 squares) background."""
    cb = np.zeros((h, w), dtype=np.uint8)

    # Define light and dark gray
    colors = [55, 255]  # dark gray, light gray

    # Fill quadrants
    half_h, half_w = h // 2, w // 2
    cb[:half_h, :half_w] = colors[0]  # top-left
    cb[:half_h, half_w:] = colors[1]  # top-right
    cb[half_h:, :half_w] = colors[1]  # bottom-left
    cb[half_h:, half_w:] = colors[0]  # bottom-right

    # Make it RGB
    cb_rgb = np.dstack([cb]*3)
    return cb_rgb


H, W = contour_rgba.shape[:2]
background = checkerboard_2x2(H, W)


# --- Blend RGBA over the background ---
alpha = contour_rgba[...,3:4] / 255.0   # (H,W,1)
overlay = (alpha * contour_rgba[...,:3] + (1-alpha)*background).astype(np.uint8)

# --- Show ---
plt.figure(figsize=(7,7))
plt.imshow(overlay)
plt.axis("off")
plt.title("Contour RGBA over checkerboard")
plt.show()



# --- Blend RGBA over the background ---
alpha = depth_rgba_transparent_bg[...,3:4] / 255.0   # (H,W,1)
overlay = (alpha * depth_rgba_transparent_bg[...,:3] + (1-alpha)*background).astype(np.uint8)

# --- Show ---
plt.figure(figsize=(7,7))
plt.imshow(overlay)
plt.axis("off")
plt.title("Depth (opaque) RGBA over checkerboard")
plt.show()


### Show semitransparent alpha overlay on depth map

Here, the transparent rgb is overlayed on the depth image creating a composition of depth map and rgb, with the semi-transparent object pixels.

In [ ]:
# depth_rgba: RGBA from your depth visualization (bg_mode="transparent")
# contour_rgba: RGBA from contour function (object semi-transparent, contour opaque)

composited = myimgtools.ImageProcessor.alpha_over_rgba(depth_rgba_transparent_bg, contour_rgba, resize_overlay=True)
# Show with matplotlib
import matplotlib.pyplot as plt
plt.figure(figsize=(6,6))
plt.imshow(composited)
plt.axis("off")
plt.title("Contour over Depth (RGBA)")
plt.show()


### Visualizing reference image and test image as overlay
This example shows the overlay of the depth+rgb with the reference image.

In [ ]:
# Create a new alpha channel with a solid value (e.g., 255 for full opacity)
alpha_channel = np.full((256, 256), fill_value=0, dtype=np.uint8)

# Use dstack to stack the alpha channel onto the existing RGB image
rgba_image1 = np.dstack((p3d_img, alpha_channel))

# Use dstack to stack the alpha channel onto the existing RGB image
rgba_image2 = np.dstack((rgb_rotated, alpha_channel))



In [ ]:

# Assume you already have your RGBA result
contour_img_ref = myimgtools.ImageProcessor.contour_from_nonwhite_rgb_semi_transparent(
    rgb_p3d,
    draw_color=(0,0,255),
    thickness=2,
    return_rgba=True,
    alpha_value=0.99
)



# Assume you already have your RGBA result
contour_rgba = myimgtools.ImageProcessor.contour_from_nonwhite_rgb_semi_transparent(
    rgb_rotated,
    draw_color=(255,0,0),
    thickness=2,
    return_rgba=True,
    alpha_value=0.7
)



# depth is a (H,W) torch tensor from PyTorch3D fragments.zbuf
# depth_rgba2 = unproject_tools.ImageProcessor.depth_to_rgb(depth_ref, cmap="gray", bg_mode="transparent")

composited2 = myimgtools.ImageProcessor.alpha_over_rgba(contour_img_ref, contour_rgba, resize_overlay=True)
# Show with matplotlib
import matplotlib.pyplot as plt
plt.figure(figsize=(6,6))
plt.imshow(composited2)
plt.axis("off")
plt.title("Contour over Depth (RGBA)")
plt.show()

